In [1]:
from architecture import ConvBlock,Encoder,Encodings,CrossAttention,SelfAttention,FNN,BLOCK,Decoder,Model,Train
from get_batches import get_batches
import torch
import torch.nn as nn

In [2]:
model=Model(ConvBlock,Encoder,Encodings,CrossAttention,SelfAttention,FNN,BLOCK,Decoder)

In [3]:
model.to('cuda').float()

Model(
  (encoder_optical): Encoder(
    (ConvBlock1): ConvBlock(
      (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (instancenorm1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (instancenorm2): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (activation): LeakyReLU(negative_slope=0.2)
    )
    (ConvBlock2): ConvBlock(
      (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (instancenorm1): InstanceNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (instancenorm2): InstanceNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=False)
      (activation): LeakyReLU(negative_slope=0.2)
    )
    (ConvBlock3): ConvBlock(
      (c

In [5]:
ckpt=torch.load('epoch25')

In [6]:
model.load_state_dict(ckpt['model'])

<All keys matched successfully>

In [7]:

from pytorch_msssim import ssim
def weighted_l1(mask,pred,target,w=5.0):
    weight=1+ (w-1)*mask
    return (weight*(pred-target).abs()).mean()

def spectral_loss(mask, pred, target, eps=1e-6):
    m = mask.expand_as(pred)
    n = m.sum(dim=(2,3)).clamp(min=1)                       
    mp = (pred*m).sum(dim=(2,3)) / n
    mt = (target*m).sum(dim=(2,3)) / n
    vp = (((pred   - mp[..., None, None])**2) * m).sum(dim=(2,3)) / n
    vt = (((target - mt[..., None, None])**2) * m).sum(dim=(2,3)) / n
    sp, st = (vp + eps).sqrt(), (vt + eps).sqrt()
    return (mp - mt).abs().mean() + (sp - st).abs().mean()



In [8]:
import numpy as np
import matplotlib.pyplot as plt

In [9]:
L=0
for i in range(1,6):
    data=np.load(f'image_0{i}.npy')
    Loss=0
    for j in range(32):
        optical=torch.from_numpy(data[8*j:8*j+8,0:3,:,:]).to('cuda').float()
        sar=torch.from_numpy(data[8*j:8*j+8:,3:5,:,:]).to('cuda').float()
        mask=torch.from_numpy(data[8*j:8*j+8,5:6,:,:]).to('cuda').float()
        input=optical*(1-mask)+mask
        with torch.no_grad():
            pred=model(input,sar)
        l1=weighted_l1(mask,pred,optical)
        ssim_loss=1-ssim(pred,optical,data_range=1.0,size_average=True)
        spectral=spectral_loss(mask,pred,optical)
        loss=l1 + 0.5*(ssim_loss) + 0.05*spectral
        Loss+=loss/32
    L+=Loss/5    

In [10]:
L

tensor(0.2557, device='cuda:0')

In [2]:
#validation loss before training-1.250
#validation loss after 200 steps-0.7958
#validation Loss after 500 steps-0.6175
#validation Loss after 1000 steos-0.5598
#validation Loss after 1 epoch - 0.5597
#validation Loss after 2 epoch-0.5707
#validation loss after 3 epoch-0.5718
#validation Loss after 4 epoch-0.5484
#validation Loss after 5 epoch-0.5231
#validation Loss after 6 epoch-0.5138
#validation Loss after 7 epoch-0.5074
#validation Loss after 10 epoch-0.4772
#validation Loss after 15 epoch-0.3910
#validation Loss after 16 epoch-0.3814
#validation Loss after 17 epoch-0.4502
#validation Loss after 18 epoch-0.3532
#validation Loss after 20 epoch-0.3278
#validation Loss after 21 epoch-0.3120
#validation Loss after 22 epoch-0.2953
#validation Loss after 23-epoch-0.2835
#validation Loss after 24-epoch-0.2704
#validation Loss after 25-epoch-0.2557
